- You have access to a collection of Twitter data and can use either RCC or GCP to complete the assignment.  This is live collection, so depending on when you complete the analysis, the results might be slightly different.  Once you combine the files, there will be over 300M records (2TB+), most of these records are related to either “Chicago” or “University”, but only a small fraction of these would be directly related to the University of Chicago.
- Your objective is to identify the profiles of Twitterers, who are tweeting about University of Chicago and compare them to the profiles of Twitterers who are tweeting about other universities.  You want to make actionable business recommendations to help University improve the social media outreach programs. (Twitterer is the name given to those who Twitter - Twitter users: https://www.merriam-webster.com/dictionary/twitterer (Links to an external site.)). 

 - 1) Identify tweets related to UChicago and 3-4 universities of your choice
   - Choose universities with sufficient tweeter activity
 - 2) Discard irrelevant tweets
   - Will be 95%+ of the data
 - 3) Complete thorough EDA to identify which variables you can use to profile the Twitterers
   - A lot of variables are poorly populated and will have to be discarded
 - 4) Identify the most prolific / influential Twitterers
   - By message volume
   - By message retweet
   - How much are they tweeting about the Universities vs. other topics?
 - 5) Where are these Twitterers located?
   - For UChicago
   - For other universities
   - Do you see any relationship between university locations and Twitterers’ locations?
   - Visualize the relationships
 - 6) What distinguishes University of Chicago Twitterers vs Twitterers who tweet about other universities
   - Visualize the trends
 - 7) What are the timelines of these tweets? Do you see significant peaks and valleys?
   - Do you see data collection gaps?
 - 8) How unique are the messages for each of these universities?
   - Are they mostly unique? Or mostly people are just copy-pasting the same text?
   - You can use something like Jaccard similarity / Cosine Similarity / Simhash / Minhash to measure uniqueness / similarity
   - Visualize message duplication (for each university – not between the universities)
   - Please note: this is not a topic modeling (LDA / LSA) – but text similarity analysis.



In [3]:
#Ensure we are using the right kernel
print(sc.version)

2.4.0-cdh6.3.0


In [4]:
import os
import shutil
import sh
from pyspark.sql.functions import *
#from pyspark.sql import functions as F
from pyspark.sql.types import *

In [9]:
import json

In [6]:
!hadoop fs -ls '/user/ivy2/Tweets/'

Java HotSpot(TM) 64-Bit Server VM warning: ignoring option MaxPermSize=512M; support was removed in 8.0
Found 30337 items
-rw-r--r--   3 ivy2 ivy2   55220293 2017-06-22 10:15 /user/ivy2/Tweets/tweets201706221015.json
-rw-r--r--   3 ivy2 ivy2   52384528 2017-06-22 11:15 /user/ivy2/Tweets/tweets201706221115.json
-rw-r--r--   3 ivy2 ivy2   56190692 2017-06-22 12:15 /user/ivy2/Tweets/tweets201706221215.json
-rw-r--r--   3 ivy2 ivy2   56992695 2017-06-22 13:15 /user/ivy2/Tweets/tweets201706221315.json
-rw-r--r--   3 ivy2 ivy2   54702790 2017-06-22 14:15 /user/ivy2/Tweets/tweets201706221415.json
-rw-r--r--   3 ivy2 ivy2   66415029 2017-06-22 15:15 /user/ivy2/Tweets/tweets201706221515.json
-rw-r--r--   3 ivy2 ivy2   63298555 2017-06-22 16:15 /user/ivy2/Tweets/tweets201706221615.json
-rw-r--r--   3 ivy2 ivy2   55417269 2017-06-22 17:15 /user/ivy2/Tweets/tweets201706221715.json
-rw-r--r--   3 ivy2 ivy2   54057246 2017-06-22 18:15 /user/ivy2/Tweets/tweets201706221815.json
-rw-r--r--   3 ivy2 ivy

-rw-r--r--   3 ivy2 ivy2   54128234 2017-06-27 12:15 /user/ivy2/Tweets/tweets201706271215.json
-rw-r--r--   3 ivy2 ivy2   51718940 2017-06-27 13:15 /user/ivy2/Tweets/tweets201706271315.json
-rw-r--r--   3 ivy2 ivy2   58784857 2017-06-27 14:15 /user/ivy2/Tweets/tweets201706271415.json
-rw-r--r--   3 ivy2 ivy2   72868708 2017-06-27 15:15 /user/ivy2/Tweets/tweets201706271515.json
-rw-r--r--   3 ivy2 ivy2   69309747 2017-06-27 16:15 /user/ivy2/Tweets/tweets201706271615.json
-rw-r--r--   3 ivy2 ivy2   57615290 2017-06-27 17:15 /user/ivy2/Tweets/tweets201706271715.json
-rw-r--r--   3 ivy2 ivy2   61543318 2017-06-27 18:15 /user/ivy2/Tweets/tweets201706271815.json
-rw-r--r--   3 ivy2 ivy2   49611495 2017-06-27 19:15 /user/ivy2/Tweets/tweets201706271915.json
-rw-r--r--   3 ivy2 ivy2   48658098 2017-06-27 20:15 /user/ivy2/Tweets/tweets201706272015.json
-rw-r--r--   3 ivy2 ivy2   48836738 2017-06-27 21:15 /user/ivy2/Tweets/tweets201706272115.json
-rw-r--r--   3 ivy2 ivy2   47703963 2017-06-27 22:

-rw-r--r--   3 ivy2 ivy2   64592036 2017-08-20 22:15 /user/ivy2/Tweets/tweets201708202215.json
-rw-r--r--   3 ivy2 ivy2   52834829 2017-08-20 23:15 /user/ivy2/Tweets/tweets201708202315.json
-rw-r--r--   3 ivy2 ivy2   53568693 2017-08-21 00:15 /user/ivy2/Tweets/tweets201708210015.json
-rw-r--r--   3 ivy2 ivy2   41820067 2017-08-21 01:15 /user/ivy2/Tweets/tweets201708210115.json
-rw-r--r--   3 ivy2 ivy2   35039221 2017-08-21 02:15 /user/ivy2/Tweets/tweets201708210215.json
-rw-r--r--   3 ivy2 ivy2   32867579 2017-08-21 03:15 /user/ivy2/Tweets/tweets201708210315.json
-rw-r--r--   3 ivy2 ivy2   31103395 2017-08-21 04:15 /user/ivy2/Tweets/tweets201708210415.json
-rw-r--r--   3 ivy2 ivy2   34213659 2017-08-21 05:15 /user/ivy2/Tweets/tweets201708210515.json
-rw-r--r--   3 ivy2 ivy2   43286833 2017-08-21 06:15 /user/ivy2/Tweets/tweets201708210615.json
-rw-r--r--   3 ivy2 ivy2   57044181 2017-08-21 07:15 /user/ivy2/Tweets/tweets201708210715.json
-rw-r--r--   3 ivy2 ivy2   67244823 2017-08-21 08:

-rw-r--r--   3 ivy2 ivy2   27411064 2017-10-21 03:15 /user/ivy2/Tweets/tweets201710210315.json
-rw-r--r--   3 ivy2 ivy2   26007653 2017-10-21 04:15 /user/ivy2/Tweets/tweets201710210415.json
-rw-r--r--   3 ivy2 ivy2   25306648 2017-10-21 05:15 /user/ivy2/Tweets/tweets201710210515.json
-rw-r--r--   3 ivy2 ivy2   29154891 2017-10-21 06:15 /user/ivy2/Tweets/tweets201710210615.json
-rw-r--r--   3 ivy2 ivy2   35952259 2017-10-21 07:15 /user/ivy2/Tweets/tweets201710210715.json
-rw-r--r--   3 ivy2 ivy2   45435702 2017-10-21 08:15 /user/ivy2/Tweets/tweets201710210815.json
-rw-r--r--   3 ivy2 ivy2   57577132 2017-10-21 09:15 /user/ivy2/Tweets/tweets201710210915.json
-rw-r--r--   3 ivy2 ivy2   69899023 2017-10-21 10:15 /user/ivy2/Tweets/tweets201710211015.json
-rw-r--r--   3 ivy2 ivy2   66668731 2017-10-21 11:15 /user/ivy2/Tweets/tweets201710211115.json
-rw-r--r--   3 ivy2 ivy2   70027792 2017-10-21 12:15 /user/ivy2/Tweets/tweets201710211215.json
-rw-r--r--   3 ivy2 ivy2   76023079 2017-10-21 13:

-rw-r--r--   3 ivy2 ivy2   88886191 2018-03-02 22:15 /user/ivy2/Tweets/tweets201803022215.json
-rw-r--r--   3 ivy2 ivy2   83964233 2018-03-02 23:15 /user/ivy2/Tweets/tweets201803022315.json
-rw-r--r--   3 ivy2 ivy2   71814197 2018-03-03 00:15 /user/ivy2/Tweets/tweets201803030015.json
-rw-r--r--   3 ivy2 ivy2   56173125 2018-03-03 01:15 /user/ivy2/Tweets/tweets201803030115.json
-rw-r--r--   3 ivy2 ivy2   48534062 2018-03-03 02:15 /user/ivy2/Tweets/tweets201803030215.json
-rw-r--r--   3 ivy2 ivy2   39435806 2018-03-03 03:15 /user/ivy2/Tweets/tweets201803030315.json
-rw-r--r--   3 ivy2 ivy2   38419037 2018-03-03 04:15 /user/ivy2/Tweets/tweets201803030415.json
-rw-r--r--   3 ivy2 ivy2   38424565 2018-03-03 05:15 /user/ivy2/Tweets/tweets201803030515.json
-rw-r--r--   3 ivy2 ivy2   47583596 2018-03-03 06:15 /user/ivy2/Tweets/tweets201803030615.json
-rw-r--r--   3 ivy2 ivy2   57829390 2018-03-03 07:15 /user/ivy2/Tweets/tweets201803030715.json
-rw-r--r--   3 ivy2 ivy2   79129151 2018-03-03 08:

...

-rw-r--r--   3 ivy2 ivy2   60872633 2019-10-19 20:15 /user/ivy2/Tweets/tweets201910192015.json
-rw-r--r--   3 ivy2 ivy2   60889606 2019-10-19 21:15 /user/ivy2/Tweets/tweets201910192115.json
-rw-r--r--   3 ivy2 ivy2   60542567 2019-10-19 22:15 /user/ivy2/Tweets/tweets201910192215.json
-rw-r--r--   3 ivy2 ivy2   61111946 2019-10-19 23:15 /user/ivy2/Tweets/tweets201910192315.json
-rw-r--r--   3 ivy2 ivy2   66220025 2019-10-20 00:15 /user/ivy2/Tweets/tweets201910200015.json
-rw-r--r--   3 ivy2 ivy2   57278308 2019-10-20 01:15 /user/ivy2/Tweets/tweets201910200115.json
-rw-r--r--   3 ivy2 ivy2   53537523 2019-10-20 02:15 /user/ivy2/Tweets/tweets201910200215.json
-rw-r--r--   3 ivy2 ivy2   47901567 2019-10-20 03:15 /user/ivy2/Tweets/tweets201910200315.json
-rw-r--r--   3 ivy2 ivy2   45832128 2019-10-20 04:15 /user/ivy2/Tweets/tweets201910200415.json
-rw-r--r--   3 ivy2 ivy2   45731542 2019-10-20 05:15 /user/ivy2/Tweets/tweets201910200515.json
-rw-r--r--   3 ivy2 ivy2   45406467 2019-10-20 06:

-rw-r--r--   3 ivy2 ivy2   37252624 2020-02-24 05:15 /user/ivy2/Tweets/tweets202002240515.json
-rw-r--r--   3 ivy2 ivy2   47634937 2020-02-24 06:15 /user/ivy2/Tweets/tweets202002240615.json
-rw-r--r--   3 ivy2 ivy2   61975319 2020-02-24 07:15 /user/ivy2/Tweets/tweets202002240715.json
-rw-r--r--   3 ivy2 ivy2   73616590 2020-02-24 08:15 /user/ivy2/Tweets/tweets202002240815.json
-rw-r--r--   3 ivy2 ivy2   72115082 2020-02-24 09:15 /user/ivy2/Tweets/tweets202002240915.json
-rw-r--r--   3 ivy2 ivy2   83212131 2020-02-24 10:15 /user/ivy2/Tweets/tweets202002241015.json
-rw-r--r--   3 ivy2 ivy2   91297552 2020-02-24 11:15 /user/ivy2/Tweets/tweets202002241115.json
-rw-r--r--   3 ivy2 ivy2   93172198 2020-02-24 12:15 /user/ivy2/Tweets/tweets202002241215.json
-rw-r--r--   3 ivy2 ivy2   88506561 2020-02-24 13:15 /user/ivy2/Tweets/tweets202002241315.json
-rw-r--r--   3 ivy2 ivy2   80890790 2020-02-24 14:15 /user/ivy2/Tweets/tweets202002241415.json
-rw-r--r--   3 ivy2 ivy2   87641879 2020-02-24 15:

-rw-r--r--   3 ivy2 ivy2  261300934 2020-06-28 19:15 /user/ivy2/Tweets/tweets202006281915.json
-rw-r--r--   3 ivy2 ivy2  228420094 2020-06-28 20:15 /user/ivy2/Tweets/tweets202006282015.json
-rw-r--r--   3 ivy2 ivy2  239214128 2020-06-28 21:15 /user/ivy2/Tweets/tweets202006282115.json
-rw-r--r--   3 ivy2 ivy2  247164696 2020-06-28 22:15 /user/ivy2/Tweets/tweets202006282215.json
-rw-r--r--   3 ivy2 ivy2  203940708 2020-06-28 23:15 /user/ivy2/Tweets/tweets202006282315.json
-rw-r--r--   3 ivy2 ivy2  158680983 2020-06-29 00:15 /user/ivy2/Tweets/tweets202006290015.json
-rw-r--r--   3 ivy2 ivy2  136981025 2020-06-29 01:15 /user/ivy2/Tweets/tweets202006290115.json
-rw-r--r--   3 ivy2 ivy2  119534121 2020-06-29 02:15 /user/ivy2/Tweets/tweets202006290215.json
-rw-r--r--   3 ivy2 ivy2   90785084 2020-06-29 03:15 /user/ivy2/Tweets/tweets202006290315.json
-rw-r--r--   3 ivy2 ivy2   75818163 2020-06-29 04:15 /user/ivy2/Tweets/tweets202006290415.json
-rw-r--r--   3 ivy2 ivy2   80385523 2020-06-29 05:

-rw-r--r--   3 ivy2 ivy2   93957875 2020-11-06 16:15 /user/ivy2/Tweets/tweets202011061615.json
-rw-r--r--   3 ivy2 ivy2   80631452 2020-11-06 17:15 /user/ivy2/Tweets/tweets202011061715.json
-rw-r--r--   3 ivy2 ivy2   62904539 2020-11-06 18:15 /user/ivy2/Tweets/tweets202011061815.json
-rw-r--r--   3 ivy2 ivy2   52110090 2020-11-06 19:15 /user/ivy2/Tweets/tweets202011061915.json
-rw-r--r--   3 ivy2 ivy2   57445388 2020-11-06 20:15 /user/ivy2/Tweets/tweets202011062015.json
-rw-r--r--   3 ivy2 ivy2   46936245 2020-11-06 21:15 /user/ivy2/Tweets/tweets202011062115.json
-rw-r--r--   3 ivy2 ivy2   42414811 2020-11-06 22:15 /user/ivy2/Tweets/tweets202011062215.json
-rw-r--r--   3 ivy2 ivy2   35075106 2020-11-06 23:15 /user/ivy2/Tweets/tweets202011062315.json
-rw-r--r--   3 ivy2 ivy2   30794040 2020-11-07 00:15 /user/ivy2/Tweets/tweets202011070015.json
-rw-r--r--   3 ivy2 ivy2   31263533 2020-11-07 01:15 /user/ivy2/Tweets/tweets202011070115.json
-rw-r--r--   3 ivy2 ivy2   35588407 2020-11-07 02:

-rw-r--r--   3 ivy2 ivy2   84187297 2021-01-18 17:15 /user/ivy2/Tweets/tweets202101181715.json
-rw-r--r--   3 ivy2 ivy2   82843376 2021-01-18 18:15 /user/ivy2/Tweets/tweets202101181815.json
-rw-r--r--   3 ivy2 ivy2   75789285 2021-01-18 19:15 /user/ivy2/Tweets/tweets202101181915.json
-rw-r--r--   3 ivy2 ivy2   81098964 2021-01-18 20:15 /user/ivy2/Tweets/tweets202101182015.json
-rw-r--r--   3 ivy2 ivy2   69518536 2021-01-18 21:15 /user/ivy2/Tweets/tweets202101182115.json
-rw-r--r--   3 ivy2 ivy2   61380774 2021-01-18 22:15 /user/ivy2/Tweets/tweets202101182215.json
-rw-r--r--   3 ivy2 ivy2   58202524 2021-01-18 23:15 /user/ivy2/Tweets/tweets202101182315.json
-rw-r--r--   3 ivy2 ivy2   48369551 2021-01-19 00:15 /user/ivy2/Tweets/tweets202101190015.json
-rw-r--r--   3 ivy2 ivy2   42786272 2021-01-19 01:15 /user/ivy2/Tweets/tweets202101190115.json
-rw-r--r--   3 ivy2 ivy2   36705161 2021-01-19 02:15 /user/ivy2/Tweets/tweets202101190215.json
-rw-r--r--   3 ivy2 ivy2   47154163 2021-01-19 03:

In [13]:
path = "hdfs:///user/ivy2/Tweets/"
filename = "*.json"
json_file = path+filename
json_file

'hdfs:///user/ivy2/Tweets/*.json'

In [12]:
tweets_df = spark.read.format('com.databricks.spark.json').\
options(header='false', inferschema='true', delimiter=',', quote='"').load(json_file)
tweets_df.limit(100000).cache()

Py4JJavaError: An error occurred while calling o113.load.
: java.lang.ClassNotFoundException: Failed to find data source: com.databricks.spark.json. Please find packages at http://spark.apache.org/third-party-projects.html
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:649)
	at org.apache.spark.sql.DataFrameReader.load(DataFrameReader.scala:194)
	at org.apache.spark.sql.DataFrameReader.load(DataFrameReader.scala:178)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at sun.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.lang.reflect.Method.invoke(Method.java:498)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:357)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.jav